<a href="https://colab.research.google.com/github/ZeninKris/autos-nl-analysis/blob/main/notebooks/05_Feature_Engineering_OCISEVI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05 — Feature Engineering: Siniestros OCISEVI

## Objetivo
Transformar los 203,890 registros individuales de siniestros
(una fila por accidente) en features agregadas por hora,
para poder unirlos al esqueleto temporal (una fila por hora).

## El problema central
El esqueleto tiene granularidad horaria.
Los siniestros tienen granularidad de evento individual.
Hay que "comprimir" los siniestros a nivel hora.

## Features que construimos
- `siniestros_hora` — cuántos accidentes ocurrieron esa hora
- `siniestros_municipios_industriales` — cuántos en Apodaca, Escobedo, García, Juárez, Pesquería
- `tiene_lesionados` — si hubo al menos un lesionado esa hora
- `tiene_fallecidos` — si hubo al menos un fallecido esa hora
- `clima_ocisevi_imputado` — condición de clima de OCISEVI, con 99s imputados

## Decisión: agregación por hora
No por día ni por municipio — por hora exacta.
Consistente con la granularidad del esqueleto temporal.

In [8]:
# ============================================================
# CELDA 1 — Carga de datos
# ============================================================

import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

RAW       = '/content/drive/MyDrive/Proyecto_ZMM/data_raw/'
PROCESSED = '/content/drive/MyDrive/Proyecto_ZMM/data_processed/'

# --- 1.1 Cargar siniestros unificados (output del Notebook 04) ---
df = pd.read_csv(PROCESSED + 'rativ_unificado.csv', low_memory=False)
print(f"Siniestros cargados: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# --- 1.2 Cargar esqueleto temporal con clima ---
esqueleto = pd.read_csv(PROCESSED + 'esqueleto_tiempo_eventos_clima.csv')
esqueleto['fecha_hora'] = pd.to_datetime(esqueleto['fecha_hora'])
print(f"Esqueleto cargado:   {esqueleto.shape[0]:,} filas × {esqueleto.shape[1]} columnas")

# --- 1.3 Vista rápida ---
print(f"\nColumnas siniestros: {df.columns.tolist()}")
print(f"\nPrimeras filas:")
print(df.head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Siniestros cargados: 203,890 filas × 16 columnas
Esqueleto cargado:   35,064 filas × 13 columnas

Columnas siniestros: ['MUNICIPIO', 'Día', 'Mes', 'Año', 'Día de la semana', 'Hora de reporte', 'Calle', 'Colonia', 'Referencia', 'Tipo de Hecho de Tránsito', 'Condición de clima', 'VEHICULO CODIFICADO', 'Causa del HT', 'Total de lesionados', 'Total de fallecidos', 'anio_fuente']

Primeras filas:
   MUNICIPIO  Día  Mes   Año  Día de la semana Hora de reporte  \
0         31    1    1  2023                 1        02:41:00   
1         31    1    1  2023                 1        06:30:00   
2         31    1    1  2023                 1        07:20:00   

                  Calle Colonia Referencia Tipo de Hecho de Tránsito  \
0  ALEJANDRO GARZA LEAL      99         99         Choque de crucero   
1  ARTURO B DE LA GARZA      99         99            Estrellamient

In [9]:
# ============================================================
# CELDA 2 — Construir fecha_hora por siniestro
# ============================================================

# --- 2.1 Parsear hora de reporte ---
# Viene como string 'HH:MM:SS' — extraemos solo la hora (granularidad horaria)
df['hora'] = pd.to_datetime(
    df['Hora de reporte'], format='%H:%M:%S', errors='coerce'
).dt.hour

# --- 2.2 Construir fecha_hora como timestamp horario ---
# Combinamos Año + Mes + Día + hora para obtener el índice temporal
df['fecha_hora'] = pd.to_datetime({
    'year':  df['Año'],
    'month': df['Mes'],
    'day':   df['Día'],
    'hour':  df['hora']
})

# --- 2.3 Diagnóstico ---
nulos_hora = df['hora'].isna().sum()
nulos_fh   = df['fecha_hora'].isna().sum()

print(f"Hora de reporte no parseada: {nulos_hora:,} ({nulos_hora/len(df)*100:.1f}%)")
print(f"fecha_hora nula:             {nulos_fh:,}  ({nulos_fh/len(df)*100:.1f}%)")

print(f"\nRango temporal de siniestros:")
print(f"  Desde: {df['fecha_hora'].min()}")
print(f"  Hasta: {df['fecha_hora'].max()}")

print(f"\nEjemplos:")
print(df[['Año','Mes','Día','Hora de reporte','hora','fecha_hora']].head(5))

Hora de reporte no parseada: 40,779 (20.0%)
fecha_hora nula:             40,779  (20.0%)

Rango temporal de siniestros:
  Desde: 2023-01-01 00:00:00
  Hasta: 2025-12-31 23:00:00

Ejemplos:
    Año  Mes  Día Hora de reporte  hora          fecha_hora
0  2023    1    1        02:41:00   2.0 2023-01-01 02:00:00
1  2023    1    1        06:30:00   6.0 2023-01-01 06:00:00
2  2023    1    1        07:20:00   7.0 2023-01-01 07:00:00
3  2023    1    1        08:32:00   8.0 2023-01-01 08:00:00
4  2023    1    1        11:07:00  11.0 2023-01-01 11:00:00


In [10]:
# ============================================================
# CELDA 3 — Diagnóstico: ¿por qué falla el 20% de horas?
# ============================================================

# --- 3.1 Ver los valores que no pudieron parsearse ---
mask_nulos = df['hora'].isna()
valores_fallidos = df.loc[mask_nulos, 'Hora de reporte'].value_counts().head(20)

print(f"Top 20 valores que no parsearon:")
print(valores_fallidos)

# --- 3.2 ¿Se concentran en algún año? ---
print(f"\nDistribución de fallas por año:")
print(df.loc[mask_nulos, 'anio_fuente'].value_counts().sort_index())

# --- 3.3 ¿Cuántos son código 99? ---
es_99 = df.loc[mask_nulos, 'Hora de reporte'].isin(['99', 99, '99:99:99'])
print(f"\nDe los {mask_nulos.sum():,} fallidos:")
print(f"  Son código 99:     {es_99.sum():,}")
print(f"  Otro formato:      {(~es_99).sum():,}")

Top 20 valores que no parsearon:
Hora de reporte
SD       3291
99        819
14:00     152
17:00     150
19:00     149
09:00     143
08:00     138
15:00     136
16:00     127
15:20     127
18:00     122
13:00     119
20:00     117
15:30     114
08:10     110
08:30     109
10:00     105
14:50     102
16:40     101
14:20     101
Name: count, dtype: int64

Distribución de fallas por año:
anio_fuente
2023    16185
2024    14249
2025    10345
Name: count, dtype: int64

De los 40,779 fallidos:
  Son código 99:     819
  Otro formato:      39,960


In [11]:
# ============================================================
# CELDA 4 — Corregir parseo de hora: recuperar formato HH:MM
# ============================================================

# --- 4.1 Función que intenta los dos formatos ---
def parsear_hora(valor):
    """Intenta HH:MM:SS primero, luego HH:MM. Devuelve la hora (int) o NaN."""
    if pd.isna(valor) or str(valor).strip() in ['99', 'SD', 'NA']:
        return np.nan
    for fmt in ['%H:%M:%S', '%H:%M']:
        try:
            return pd.to_datetime(str(valor).strip(), format=fmt).hour
        except ValueError:
            continue
    return np.nan

# --- 4.2 Re-parsear la columna hora con la función corregida ---
df['hora'] = df['Hora de reporte'].apply(parsear_hora)

# --- 4.3 Reconstruir fecha_hora ---
df['fecha_hora'] = pd.to_datetime({
    'year':  df['Año'],
    'month': df['Mes'],
    'day':   df['Día'],
    'hour':  df['hora']
}, errors='coerce')

# --- 4.4 Diagnóstico final ---
nulos = df['fecha_hora'].isna().sum()
print(f"Registros sin fecha_hora válida: {nulos:,} ({nulos/len(df)*100:.1f}%)")
print(f"Registros recuperados: {40779 - nulos:,}")
print(f"Rango temporal:")
print(f"  Desde: {df['fecha_hora'].min()}")
print(f"  Hasta: {df['fecha_hora'].max()}")

Registros sin fecha_hora válida: 4,327 (2.1%)
Registros recuperados: 36,452
Rango temporal:
  Desde: 2023-01-01 00:00:00
  Hasta: 2025-12-31 23:00:00


In [12]:
# ===========================================================
# CELDA 5 (Corregida) — Limpieza de nulos irrecuperables y flag espacial
# ===========================================================

# 1. Eliminar filas donde no pudimos rescatar la fecha_hora (los SD/99 temporales)
print(f"Total de registros ANTES de limpiar nulos temporales: {len(df)}")
df = df.dropna(subset=['fecha_hora']).copy()
print(f"Total de registros DESPUÉS: {len(df)}")

# 2. Crear flag 'tiene_coordenadas' basándonos en la columna 'Referencia'
# Es válido si no es nulo y no es el código '99' (ni en texto ni en número)
df['tiene_coordenadas'] = df['Referencia'].notna() & (df['Referencia'] != '99') & (df['Referencia'] != 99)

# 3. Ver cómo se distribuyen las coordenadas por año
print("\nDesglose de registros CON coordenadas por año:")
print(df.groupby('Año')['tiene_coordenadas'].value_counts().unstack().fillna(0))

Total de registros ANTES de limpiar nulos temporales: 203890
Total de registros DESPUÉS: 199563

Desglose de registros CON coordenadas por año:
tiene_coordenadas    False    True 
Año                                
2023               55829.0  15261.0
2024                   0.0  66681.0
2025               27107.0  34685.0
